## Geophysical forward modelling: electrical resistivity tomography (ERT)

This notebook shows how to use a GemPy model as the basis for a geophysical simulation.

In **electrical resistivity tomography (ERT)**, an electrical current is injected into the ground through two electrodes, and the resulting voltage is measured between two other electrodes. Repeating this for many electrode combinations gives *apparent resistivities*, which depend on the resistivity distribution in the subsurface, and thus on the geology.

Here we compute such data for our structural model (**forward modelling**): we assign a resistivity to every lithology, place electrodes along a profile on the surface and simulate the measurements with [pyGIMLi](https://www.pygimli.org/). Synthetic data like this are useful to test whether a survey could resolve the structures of interest, and they are the starting point for inversion (see the outlook at the end).

**Workflow in this notebook:**
1. Build a structural model with GemPy
2. Transfer the lithologies to a pyGIMLi mesh
3. Assign resistivities
4. Extend the mesh with a boundary zone
5. Place electrodes and define measurement schemes
6. Simulate the ERT data

In addition to GemPy, it requires [pyGIMLi](https://www.pygimli.org/) and [PyVista](https://pyvista.org/). The 3D plots open in separate windows, so this notebook is meant to run locally. On macOS, install pyGIMLi with conda (see the README).

In [ ]:
# Importing necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import pyvista as pv

import gempy as gp
import gempy_viewer as gpv

import pygimli as pg
import pygimli.physics.ert as ert

### 1) Structural geological model

The same model as in the step-by-step example (three layers and a basement, offset by a fault), with a coarser resolution of 25 x 25 x 25 cells to keep the simulation fast.

In [ ]:
# Model extent and resolution
xmin, xmax = 0, 2500
ymin, ymax = 0, 1000
zmin, zmax = 0, 1000
nx, ny, nz = 25, 25, 25

data_path = 'https://raw.githubusercontent.com/cgre-aachen/gempy_data/master/'
path_to_data = data_path + "/data/input_data/jan_models/"

# Create a GeoModel instance
geo_model = gp.create_geomodel(
    project_name='combination',
    extent=[xmin, xmax, ymin, ymax, zmin, zmax],
    resolution=[nx, ny, nz],
    importer_helper=gp.data.ImporterHelper(
        path_to_orientations=path_to_data + "model7_orientations.csv",
        path_to_surface_points=path_to_data + "model7_surface_points.csv"
    )
)

# Map geological series to surfaces
gp.map_stack_to_surfaces(
    gempy_model=geo_model,
    mapping_object={
        "Fault_Series": ('fault'),
        "Strat_Series1": ('rock3'),
        "Strat_Series2": ('rock2', 'rock1'),
    }
)
# Define youngest structural group as fault
gp.set_is_fault(geo_model, ["Fault_Series"])

# Compute the geological model
gp.compute_model(geo_model)

In [ ]:
# 2D cross-section with the model cells as grid
p2d = gpv.plot_2d(geo_model, show=False)
p2d.axes[0].minorticks_on()
p2d.axes[0].grid(which='major', linestyle='--', linewidth=0.8, color='black')
p2d.axes[0].grid(which='minor', linestyle=':', linewidth=0.4, color='black')
plt.show()

### 2) Transfer the lithologies to a pyGIMLi mesh

pyGIMLi needs its own mesh. We create a regular grid with exactly the same cells as the GemPy model: its nodes are the cell corners, so there is one more node than cells in each direction.

GemPy and pyGIMLi number the cells in a different order (GemPy: z changes fastest, pyGIMLi: x changes fastest), so the lithology array has to be reordered.

In [ ]:
# Cell corners (nodes) of the GemPy grid
gridx = np.linspace(xmin, xmax, nx + 1)
gridy = np.linspace(ymin, ymax, ny + 1)
gridz = np.linspace(zmin, zmax, nz + 1)

# Regular pyGIMLi grid with the same cells
grid = pg.createGrid(gridx, gridy, gridz)
print(grid)

In [ ]:
# Lithology id per cell, reordered from GemPy (z fastest) to pyGIMLi (x fastest)
litho = geo_model.solutions.raw_arrays.lith_block.reshape([nx, ny, nz]).T.flatten(order="C")

# Check: lithologies on the pyGIMLi mesh
pg.show(grid, litho, showMesh=True, cMap='Spectral', cMin=1, cMax=5, label='Lithology id')

### 3) Resistivity model

Every lithology gets a resistivity (in Ωm). The values are just an example: a conductive layer (`rock2`) between more resistive units.

In [ ]:
# Resistivity per lithology (Ωm)
lith_resistivity = {
    'rock3': 1000,
    'rock2': 10,
    'rock1': 500,
    'basement': 2100,
}

# Lithology id = position in the structural frame + 1
element_names = geo_model.structural_frame.elements_names
resistivities = np.zeros_like(litho, dtype=float)
for name, rho in lith_resistivity.items():
    resistivities[litho == element_names.index(name) + 1] = rho

assert np.all(resistivities > 0), "Some cells have no resistivity assigned"

In [ ]:
# Vertical slice through the resistivity model (y = 500 m)
meshpv = pg.viewer.pv.pgMesh2pvMesh(grid, resistivities, label='Resistivity (Ωm)')

pl = pv.Plotter(notebook=False)
pl.add_mesh(meshpv.outline(), color='k')
pl.add_mesh(meshpv.slice(normal=[0, 1, 0]), show_edges=True, cmap='Spectral', log_scale=True)
pl.show_axes()
pl.show()

### 4) Boundary zone around the mesh

The electrical current spreads out far into the ground. If the model boundaries are close to the electrodes, they distort the simulated potentials. We therefore extend the mesh with a zone of increasingly large cells, out to 4 km on all sides and below. The resistivities in this zone are continued from the nearest model cell.

Without this zone, the simulated data are clearly inaccurate: normal and reciprocal measurements (current and potential electrodes swapped), which should give identical results, differ by more than 20 %. With the boundary zone, the difference drops below 5 %.

In [ ]:
# Distances (m) of the additional node planes from the model edge,
# added on all sides and below, not above the surface
padding = np.array([100, 250, 500, 1000, 2000, 4000], dtype=float)

px = np.concatenate([xmin - padding[::-1], gridx, xmax + padding])
py = np.concatenate([ymin - padding[::-1], gridy, ymax + padding])
pz = np.concatenate([zmin - padding[::-1], gridz])

mesh = pg.createGrid(px, py, pz)
print(mesh)

In [ ]:
# Resistivity of every cell: value of the nearest cell of the original model
centers = np.array([cell.center().array() for cell in mesh.cells()])
dx, dy, dz = (xmax - xmin) / nx, (ymax - ymin) / ny, (zmax - zmin) / nz
ix = np.clip(((centers[:, 0] - xmin) // dx).astype(int), 0, nx - 1)
iy = np.clip(((centers[:, 1] - ymin) // dy).astype(int), 0, ny - 1)
iz = np.clip(((centers[:, 2] - zmin) // dz).astype(int), 0, nz - 1)
mesh_resistivities = resistivities[ix + iy * nx + iz * nx * ny]

# Boundary conditions: mixed boundary condition (marker -2) on all outer faces except the surface
for boundary in mesh.boundaries():
    if boundary.outside() and boundary.center().z() < zmax:
        boundary.setMarker(-2)

### 5) Electrodes and measurement schemes

We place 24 electrodes along a profile in x-direction on the surface (100 m to 2400 m, spacing 100 m). The electrodes have to sit on mesh nodes: the nodes are 100 m apart in x and 40 m in y, so the profile runs along y = 520 m.

In [ ]:
# Electrode positions on the surface
sensors_x = np.arange(100, 2401, 100.0)
sensors = np.column_stack([sensors_x,
                           np.full_like(sensors_x, 520.0),
                           np.full_like(sensors_x, zmax)])

# Check that every electrode coincides with a mesh node
node_pos = np.array([node.pos().array() for node in mesh.nodes()])
on_node = np.array([np.any(np.all(np.isclose(node_pos, s), axis=1)) for s in sensors])
print(f"{on_node.sum()} of {len(sensors)} electrodes are on mesh nodes")

In [ ]:
# Electrodes on top of the resistivity model
pl = pv.Plotter(notebook=False)
pl.add_mesh(meshpv, show_edges=True, cmap='Spectral', log_scale=True)
pl.add_points(sensors, color='red', point_size=10, render_points_as_spheres=True)
pl.show_axes()
pl.show()

A measurement **scheme** defines which electrodes are used for current injection and which for the voltage measurement. We combine three common configurations: dipole-dipole (`dd`), Schlumberger (`slm`) and Wenner (`wa`).

In [ ]:
# Create the measurement schemes (along the profile) and set the 3D electrode positions
schemes = []
for name in ['dd', 'slm', 'wa']:
    scheme = ert.createData(elecs=sensors[:, 0], schemeName=name, addInverse=True)
    for i, pos in enumerate(sensors):
        scheme.setSensorPosition(i, pos)
    schemes.append(scheme)
    print(f"{name}: {scheme.size()} measurements")

# Combine them into one scheme
scheme = schemes[0]
for other in schemes[1:]:
    scheme.add(other)
print(f"Total: {scheme.size()} measurements")

### 6) Simulate the ERT data

`ert.simulate` solves the electrical potential on the mesh for every measurement and returns the apparent resistivities, here with 0.1 % noise added. This takes a few seconds.

In [ ]:
data = ert.simulate(mesh=mesh, scheme=scheme, res=mesh_resistivities,
                    noiseLevel=0.001, noiseAbs=0, seed=48)

# Remove (rare) unphysical negative apparent resistivities
data.remove(data['rhoa'] < 0)
print(data)

In [ ]:
# Apparent resistivities of all configurations (pseudo-sections)
pg.show(data, logScale=True)

### Outlook: structure-based inversion

Forward modelling computes data from a model. **Inversion** goes the other way: it searches for a model that explains measured data. In a structure-based inversion, the parameters of the GemPy model itself (e.g. surface points) are updated until the simulated data fit the observations, so the result is again a consistent geological model.

The following example uses travel time (seismic) data instead of ERT:

![synthetic_study](images/synthetic_study.png)

![structural_inversion](images/structural_inversion.gif "structural inversion")

(Balza et al., publication in progress)